# Day 3 (Colab) — QLoRA Training, cars_lite

Lite config per `autopricer_instructions_v2.md`: `r=32`, attention-only LoRA, 1 epoch.

**T4-specific fixes applied in this version** (discovered via debugging):
- `bnb_4bit_compute_dtype=torch.float16` (not bf16 — T4 lacks bf16 tensor cores, causes ~100x slowdown)
- `dtype=torch.float16` at model load (forces non-quantized layers to fp16 too, avoids dtype mismatch)
- `fp16=True` **and** `bf16=False` explicitly in `SFTConfig` (bf16 silently autodetects to `True` otherwise, even with fp16 set)
- `group_by_length` removed (not supported in `trl` 1.7.0 — replaced by `packing`, not used here)
- `report_to="none"`, `push_to_hub=False` — kept off for this first verification run

## Cell 1: Install dependencies, auth, load tokenizer + 4-bit model + dataset

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes datasets peft trl

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset

login(token=userdata.get('HF_TOKEN'))

BASE_MODEL = "meta-llama/Llama-3.2-3B"
HF_USER = "ShahidHKhan"
PROJECT_NAME = "autopricer"

LITE_MODE = True
DATASET_NAME = f"{HF_USER}/cars_lite_prompts" if LITE_MODE else f"{HF_USER}/cars_full_prompts"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4 fix: not bfloat16
    bnb_4bit_quant_type="nf4"
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=quant_config,
    dtype=torch.float16,  # T4 fix: forces non-quantized layers to fp16 too
)

ds = load_dataset(DATASET_NAME)
train = ds["train"]
val = ds["val"]

print(f"Train: {len(train)}, Val: {len(val)}")
print(f"Model dtype check — first param: {next(base_model.parameters()).dtype}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.0 MB/s eta 0:00:00


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/123k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26132 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1452 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1452 [00:00<?, ? examples/s]

Train: 26132, Val: 1452
Model dtype check — first param: torch.float16


**Expected:** `Train: 26132, Val: 1452`, and the dtype check should print `torch.float16` (not `torch.bfloat16`) — confirms the fix landed before we build the trainer.

Stop here and verify this output before running Cell 2.

## Cell 2: LoRA config + SFTTrainer setup

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

LORA_R = 32
LORA_ALPHA = LORA_R * 2
LORA_DROPOUT = 0.1
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
TARGET_MODULES = ATTENTION_LAYERS  # lite mode: attention only

EPOCHS = 1
BATCH_SIZE = 32
MAX_SEQUENCE_LENGTH = 128
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = "cosine"
OPTIMIZER = "paged_adamw_32bit"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-lite"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    optim=OPTIMIZER,
    learning_rate=LEARNING_RATE,
    fp16=True,
    bf16=False,  # T4 fix: must be explicit, bf16 autodetects True otherwise
    max_grad_norm=0.3,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    max_length=MAX_SEQUENCE_LENGTH,
    report_to="none",   # no W&B key set up yet
    push_to_hub=False,  # verify training works locally-in-Colab first
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,
)

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters,
)

print("Trainer ready.")
print(f"Total training steps: {len(train) // BATCH_SIZE * EPOCHS}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/26132 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/26132 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/26132 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1452 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1452 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1452 [00:00<?, ? examples/s]

Trainer ready.
Total training steps: 816


**Expected:** `Trainer ready.` and `Total training steps: 816`.

In [ ]:
# Fix: LoRA (trainable) params must be fp32 for AMP's GradScaler — only the
# frozen base model should be fp16. Casting trainable params to fp32 instead.
cast_count = 0
for name, param in fine_tuning.model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.to(torch.float32)
        cast_count += 1

print(f"Cast {cast_count} trainable parameters to fp32.")

mismatches = [name for name, param in fine_tuning.model.named_parameters()
              if param.requires_grad and param.dtype != torch.float32]
print(f"Remaining trainable-param mismatches: {len(mismatches)}")

Cast 224 trainable parameters to fp32.
Remaining trainable-param mismatches: 0


## Cell 3: Train

Free Colab T4 compute, no API cost. Watch the first ~10-20 steps for a sane it/s (expect roughly 1-3 it/s, i.e. a few minutes to ~15 min total — **not** the 0.04 it/s / 6-hour ETA we saw before the fixes). If it's still that slow, stop and paste output before letting it run further.

In [ ]:
fine_tuning.train()

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.435908,1.434069,1.377643,130272.000000,0.685440
100,1.422353,1.410049,1.505838,256087.000000,0.691484
150,1.393778,1.385340,1.413201,381665.000000,0.693544
200,1.429087,1.375188,1.394985,506797.000000,0.694368
250,1.399311,1.364632,1.432684,632682.000000,0.692995
300,1.362183,1.347577,1.372978,758176.000000,0.694231
350,1.358944,1.339161,1.365642,884037.000000,0.693544
400,1.343718,1.336332,1.294574,1009737.000000,0.695467
450,1.330812,1.327967,1.347832,1134866.000000,0.698214
500,1.346751,1.323443,1.352282,1260379.000000,0.697115


TrainOutput(global_step=817, training_loss=1.3715676631017006, metrics={'train_runtime': 4831.9619, 'train_samples_per_second': 5.408, 'train_steps_per_second': 0.169, 'total_flos': 4.421997716012237e+16, 'train_loss': 1.3715676631017006, 'epoch': 1.0})

In [ ]:
PROJECT_RUN_NAME = f"{PROJECT_NAME}-lite"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

fine_tuning.model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)

print(f"Pushed to: https://huggingface.co/{HUB_MODEL_NAME}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 16.7kB / 73.4MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpdekuavlk/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Pushed to: https://huggingface.co/ShahidHKhan/autopricer-lite
